In [1]:
import glob
import pandas as pd
import numpy as np
import os
import pickle

## This code scales data from intan voltage (0 - 3.3V) to the applied voltage (±5V) and converts it to degree s

In [2]:
out_path = "./dataFolder/4_RawData_Index_Timestamp/ScaledStimulus/"
os.makedirs(out_path, exist_ok=True)

In [3]:
# save all scaling infor into a readMe for easy lookup

#scaling- ±5V motor signal was converted to 0-3.3V by hardware circuit before recording (adjust accordinly for different setup)
ADC_RANGE = 3.3            # ADC full range (0 to 3.3V - the one we saw)
ADC_MIDPOINT = ADC_RANGE/2     # 0V motor == 1.65V ADC (fixed hardware midpoint, now stimulus should go from -1.65 to + 1.65)
MOTOR_RANGE = 10.0         # ±5V = 10V full range
scale_factor = MOTOR_RANGE / ADC_RANGE  # ~3.03

scaling_info = pd.DataFrame({
    'parameter': ['ADC_midpoint', 'ADC_range', 'motor_range', 'scale_factor'],
    'value':     [ADC_MIDPOINT,   ADC_RANGE,   MOTOR_RANGE,   scale_factor],
    'units':     ['V',            'V',          'V',           'V/V']})
scaling_info.to_csv(out_path + 'scaling_info.csv', index=False)

In [4]:
# get the stimulus files

stim_files = glob.glob("./dataFolder/4_RawData_Index_Timestamp/*.pickle")
print(f'Found {len(stim_files)} files')

Found 4 files


In [5]:
for filepath in stim_files:
    print(f'Processing: {filepath}')
    
    name = f"_".join(os.path.basename(filepath).split('_')[0:2])
    
    with open(filepath, "rb") as f:
        data = pickle.load(f)
    
    stimulus = data[2]
    # Subtract fixed hardware midpoint (1.65V) then scale back to ±5V
    stimulus_scaled = (stimulus - ADC_MIDPOINT) * scale_factor
    
    # center the stimulus to the mean of the data - assuming proboscis was aligned when we started
    stimulus_scaled = stimulus_scaled - np.mean(stimulus_scaled)

    # Save individually per moth
    all_Units_aligned = data[0]
    all_Units_prior = data[1]
    WhiteNoise_scaled = stimulus_scaled
    saveObject = (all_Units_aligned,all_Units_prior, WhiteNoise_scaled)
    with open(out_path + name + '_timeStamps_scaled.pickle',"wb") as f:
        pickle.dump(saveObject, f)
    
#     moth_df.to_csv(out_path + name + '_Displacement_scaledInVolts.csv', index=False)
    print(f'  Saved: {name}_scaled_voltage.csv')
    print(f'  Voltage range: {stimulus_scaled.min():.4f} to {stimulus_scaled.max():.4f} V')

print('All done!')

Processing: ./dataFolder/4_RawData_Index_Timestamp/m30_wn1_timeStamps.pickle
  Saved: m30_wn1_scaled_voltage.csv
  Voltage range: -3.6674 to 2.9774 V
Processing: ./dataFolder/4_RawData_Index_Timestamp/m29_wn2_timeStamps.pickle
  Saved: m29_wn2_scaled_voltage.csv
  Voltage range: -2.6470 to 2.3126 V
Processing: ./dataFolder/4_RawData_Index_Timestamp/m23_wn2_timeStamps.pickle
  Saved: m23_wn2_scaled_voltage.csv
  Voltage range: -3.6433 to 2.9288 V
Processing: ./dataFolder/4_RawData_Index_Timestamp/m31_wn2_timeStamps.pickle
  Saved: m31_wn2_scaled_voltage.csv
  Voltage range: -2.6664 to 2.3029 V
All done!
